In [2]:
import math
from functools import partial


%load_ext autoreload
%autoreload 2
from benchmark_tool import run_benchmark, run_benchmark_as_subprocess, serial, threaded, multiprocessing

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
%%writefile task.py

import math
def cpu_task(n):
    """A CPU-Bound task--no real memory usage, no waiting."""
    x = 0
    for i in range(n):
        x += math.sqrt(i * 10 + 5.2)



Overwriting task.py


In [13]:
run_benchmark_as_subprocess('task:cpu_task', params=dict(n=50000), n_repeats=100, executor_entry_point='serial')

{'run_id': '4df0858e',
 'wall': 0.4806,
 'process': 0.4688,
 'cpu_percent': 96.6,
 'io_read_count': 0,
 'io_read_bytes': 0,
 'io_read_rate': 0.0,
 'io_write_count': 0,
 'io_write_bytes': 0,
 'io_write_rate': 0.0}

In [15]:
run_benchmark_as_subprocess('task:cpu_task', params=dict(n=50000), n_repeats=100, executor_entry_point='multiprocessing')

{'run_id': '9f7294af',
 'wall': 0.2705,
 'process': 0.5469,
 'cpu_percent': 5.6,
 'io_read_count': 519,
 'io_read_bytes': 52122,
 'io_read_rate': 192693.670601,
 'io_write_count': 209,
 'io_write_bytes': 31212,
 'io_write_rate': 115389.947562}

In [17]:
import math
def cpu_task(n):
    """A CPU-Bound task--no real memory usage, no waiting."""
    x = 0
    for i in range(n):
        x += math.sqrt(i * 10 + 5.2)

In [18]:
run_benchmark(partial(cpu_task, n=10000), n_repeats=1000, executor=threaded)

{'run_id': 'a2a1de82',
 'wall': 0.9322,
 'process': 0.9375,
 'cpu_percent': 101.7,
 'io_read_count': 1,
 'io_read_bytes': 0,
 'io_read_rate': 0.0,
 'io_write_count': 13,
 'io_write_bytes': 25124,
 'io_write_rate': 26951.338481}

In [21]:
run_benchmark(partial(cpu_task, n=10000), n_repeats=1000, executor=multiprocessing)

ValueError: multiproccesing requires subprocess support. Use run_benchmark_as_subprocess() instead.

In [94]:
import time
import pandas as pd

def io_task(n):
    """An IO-Bound task--just waiting."""
    for _ in range(n):
        time.sleep(0.001)



In [95]:
t0 = time.perf_counter()
serial(partial(io_task, 1000), n_repeats=1)
time.perf_counter() - t0

1.5761724000039976

In [104]:

run_benchmark(partial(io_task, n=100), executor=serial)

{'run_id': '0cad954f',
 'wall': 0.1593,
 'process': 0.0156,
 'cpu_percent': 10.0,
 'io_read_count': 1,
 'io_read_bytes': 0,
 'io_read_rate': 0.0,
 'io_write_count': 13,
 'io_write_bytes': 25124,
 'io_write_rate': 157681.051727}

In [77]:
run_benchmark(partial(io_task, n=100), executor=threaded)

{'run_id': '045cde1a',
 'wall': 0.1514,
 'process': 0.0,
 'cpu_percent': 0.0,
 'io_read_count': 1,
 'io_read_bytes': 0,
 'io_read_rate': 0.0,
 'io_write_count': 14,
 'io_write_bytes': 25124,
 'io_write_rate': 165929.941423}

In [83]:
run_benchmark(partial(io_task, n=1), n_repeats=100, executor=threaded)

{'run_id': '6122a267',
 'wall': 0.0216,
 'process': 0.0,
 'cpu_percent': 0.0,
 'io_read_count': 1,
 'io_read_bytes': 0,
 'io_read_rate': 0.0,
 'io_write_count': 13,
 'io_write_bytes': 25124,
 'io_write_rate': 1163870.178337}

In [21]:
%%writefile task.py

import math

def cpu_task(n):
    """A CPU-Bound task--no real memory usage, no waiting."""
    x = 0
    for i in range(n):
        x += math.sqrt(i * 10 + 5.2)

Overwriting task.py


In [31]:
run_benchmark_as_subprocess('task:cpu_task', params=dict(n=500000), n_repeats=10)

{'run_id': '45bebcfc',
 'wall': 0.4544,
 'process': 0.4375,
 'cpu_percent': 96.6,
 'io_read_count': 0,
 'io_read_bytes': 0,
 'io_read_rate': 0.0,
 'io_write_count': 0,
 'io_write_bytes': 0,
 'io_write_rate': 0.0}

In [32]:
import time
import pandas as pd

def io_task(n):
    """An IO-Bound task--just waiting."""
    with open('mydata.txt', 'wb') as f:
        f.write(b"i" * 100)

pd.DataFrame([run_benchmark(io_task, dict(n=50), n_repeats=1000) for _ in range(10)])

TypeError: 'dict' object is not callable

In [33]:
0.1719 / 0.1672

1.02811004784689

In [ ]:

def numpy_task(n):
    """A CPU-Bound task done out-of-Python."""
    start_cpu = time.process_time()
    a = np.random.rand(n, n)
    b = np.random.rand(n, n)
    np.matmul(a, b)
    return time.process_time() - start_cpu


Signature:
run_benchmark(
    task: Callable,
    batch_executor: benchmark_tool.Executor,
    *args,
    **kwargs,
) -> dict[str, 'JSON']
Docstring: Runs code and measures elapsed time.
File:      c:\users\delgr\projects\hhai-repo\notebooks\04_profiling\benchmark_tool.py
Type:      function